# Точечная проверка полей: 5 ИНН или 5 номеров договоров

Назначение:
- быстро проверить на малом периметре, что поля подтягиваются корректно;
- собрать атрибуты из источников и показать итоговый merged-слой;
- увидеть null/проблемные поля до запуска полного расчета.


In [ ]:
import re
from decimal import Decimal, InvalidOperation
import pandas as pd
from rail_connectors.connection import connect

pd.options.display.max_columns = None
pd.options.display.width = None
pd.options.display.max_colwidth = None


In [ ]:
# Параметры запуска
month_start = '2026-02-01'
month_end = (pd.to_datetime(month_start) + pd.offsets.MonthEnd(0)).strftime('%Y-%m-%d')
aur_rate = 1926.0

# Заполните до 5 значений в любом из списков
target_inn_list = [
    # '7707083893',
]
target_contract_list = [
    # 'ЕЭД250361/00073',
]

print('month_start =', month_start)
print('month_end =', month_end)
print('target_inn_list size =', len(target_inn_list))
print('target_contract_list size =', len(target_contract_list))

if len(target_inn_list) == 0 and len(target_contract_list) == 0:
    raise ValueError('Заполните target_inn_list или target_contract_list (до 5 значений).')
if len(target_inn_list) > 5 or len(target_contract_list) > 5:
    raise ValueError('Для этой тетрадки используйте не более 5 ИНН и/или 5 договоров.')


In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()

with imp:
    imp.execute('invalidate metadata ods_alpha.scd1_agreements')
    imp.execute('invalidate metadata ods_alpha.scd1_companies')
    imp.execute('invalidate metadata ods_alpha.scd1_merchants')
    imp.execute('invalidate metadata ods_alpha.scd1_pos_terminals')
    imp.execute('invalidate metadata ods_alpha.scd1_trx')
    imp.execute('invalidate metadata ods_alpha.scd1_trx_acq')
    imp.execute('invalidate metadata ods_alpha.scd1_trx_int')
    imp.execute('invalidate metadata ods.scd1_z_r2_ip_merchants')
    imp.execute('invalidate metadata ods.scd1_z_r2_tariff_tune')
    imp.execute('invalidate metadata ods.scd1_z_r2_tariff_fix')
    imp.execute('invalidate metadata ods.scd1_z_r2_tariff_plan')
    imp.execute('invalidate metadata ods.scd1_z_depart')
    imp.execute('invalidate metadata ods.scd1_z_branch')
    imp.execute('invalidate metadata ods.scd1_z_cl_corp')
    imp.execute('invalidate metadata sandbox_ai.shestopalov_terminal_amortization_oneoff')

print('Impala initialized')


In [ ]:
def normalize_numeric_str(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace(' ', '').replace('\xa0', '')
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return None
    s = s.replace(',', '.')
    if re.search(r"[eE][+-]?\d+$", s):
        try:
            s = format(Decimal(s), 'f')
        except InvalidOperation:
            return None
    s = re.sub(r"\.0+$", '', s)
    s = re.sub(r"\D", '', s)
    return s if s else None

def normalize_inn(v):
    s = normalize_numeric_str(v)
    if s is None:
        return None
    if len(s) in (10, 12):
        return s
    if len(s) == 9:
        return s.zfill(10)
    if len(s) == 11:
        return s.zfill(12)
    return None

def normalize_contract(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', ' ')
    s = re.sub(r"\s+", ' ', s)
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return None
    return s.lower()

def normalize_agr_id(v):
    return normalize_numeric_str(v)

def to_sql_in_list(values):
    vals = [str(v) for v in values if pd.notna(v) and str(v).strip() != '']
    if not vals:
        return None
    return ', '.join([f"'{v}'" for v in vals])


## 1) Base-периметр по 5 ИНН/договорам


In [ ]:
inn_norm_list = [normalize_inn(x) for x in target_inn_list if normalize_inn(x) is not None]
contract_norm_list = [normalize_contract(x) for x in target_contract_list if normalize_contract(x) is not None]

inn_filter = f"case when length(regexp_replace(trim(c.c_inn), '[^0-9]', '')) = 9 then lpad(regexp_replace(trim(c.c_inn), '[^0-9]', ''), 10, '0') when length(regexp_replace(trim(c.c_inn), '[^0-9]', '')) = 11 then lpad(regexp_replace(trim(c.c_inn), '[^0-9]', ''), 12, '0') else regexp_replace(trim(c.c_inn), '[^0-9]', '') end"
contract_filter = "lower(trim(cast(a.c_agr_number as string)))"

cond_parts = []
if inn_norm_list:
    cond_parts.append(f"{inn_filter} in ({to_sql_in_list(inn_norm_list)})")
if contract_norm_list:
    cond_parts.append(f"{contract_filter} in ({to_sql_in_list(contract_norm_list)})")

where_target = ' or '.join(cond_parts)

sql_base = f"""
select
    {inn_filter} as inn_norm,
    cast(c.c_cmp_name as string) as client_name,
    {contract_filter} as contract_norm,
    cast(a.c_agr_number as string) as contract_number,
    cast(a.abs_agr_id as string) as agr_id_norm,
    cast(a.n_agr as string) as n_agr,
    cast(a.d_valid_from as date) as d_valid_from,
    cast(a.d_valid_to as date) as d_valid_to,
    cast(a.n_cmp_client as string) as n_cmp_client
from ods_alpha.scd1_agreements a
join ods_alpha.scd1_companies c on c.n_cmp = a.n_cmp_client
where a.acq_class = 'SA'
  and cast(a.d_valid_from as date) <= cast('{month_start}' as date)
  and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
  and coalesce(a.ods_deleted_flg, '0') <> '1'
  and coalesce(c.ods_deleted_flg, '0') <> '1'
  and ({where_target})
"""

with imp:
    base_df = imp.fetch(sql_base)

base_df['inn_norm'] = base_df['inn_norm'].apply(normalize_inn)
base_df['contract_norm'] = base_df['contract_norm'].apply(normalize_contract)
base_df['agr_id_norm'] = base_df['agr_id_norm'].apply(normalize_agr_id)

display(base_df)
print('base_rows =', len(base_df))


## 2) R2-поля (ОГРН / РФ / ВСП / тариф / фикс)


In [ ]:
agr_ids = sorted(base_df['agr_id_norm'].dropna().unique().tolist())
if not agr_ids:
    r2_df = pd.DataFrame(columns=['agr_id_norm'])
else:
    in_list = to_sql_in_list(agr_ids)
    sql_r2 = f"""
    with r2 as (
        select cast(m.id as string) as agr_id_norm, m.c_cl_org, m.c_depart, m.c_tariff_plan
        from ods.scd1_z_r2_ip_merchants m
        where cast(m.id as string) in ({in_list})
    ),
    fix_r2 as (
        select
            cast(m.id as string) as agr_id_norm,
            max(cast(tf.c_summa as double)) as commission_monthly,
            count(distinct cast(tf.c_summa as string)) as fix_variants_cnt
        from ods.scd1_z_r2_ip_merchants m
        left join ods.scd1_z_r2_tariff_tune tt on tt.c_tariff_plan = m.c_tariff_plan
        left join ods.scd1_z_r2_tariff_fix tf on tt.c_tariff = tf.id
        where cast(m.id as string) in ({in_list})
        group by cast(m.id as string)
    )
    select
        r2.agr_id_norm,
        corp.c_register_gos_reg_num_rec as ogrn,
        dep.c_name as vsp_name,
        dep.c_num as vsp_code,
        br.c_shortlabel as filial_rf,
        tp.c_name as tariff_name,
        fr.commission_monthly,
        fr.fix_variants_cnt
    from r2
    left join ods.scd1_z_cl_corp corp on corp.id = r2.c_cl_org
    left join ods.scd1_z_depart dep on dep.id = r2.c_depart
    left join ods.scd1_z_branch br on br.id = dep.c_filial
    left join ods.scd1_z_r2_tariff_plan tp on tp.id = r2.c_tariff_plan
    left join fix_r2 fr on fr.agr_id_norm = r2.agr_id_norm
    """
    with imp:
        r2_df = imp.fetch(sql_r2)

display(r2_df)
print('r2_rows =', len(r2_df))


## 3) Company-атрибуты (MCC / точки / терминалы / амортизация)


In [ ]:
n_cmps = sorted(base_df['n_cmp_client'].dropna().unique().tolist())
if not n_cmps:
    cmp_df = pd.DataFrame(columns=['n_cmp_client'])
else:
    in_list = ', '.join(n_cmps)
    sql_cmp = f"""
    with cmp_retl as (
        select
            cast(m.n_cmp as string) as n_cmp_client,
            count(distinct cast(m.c_nmrc as string)) as retl_cnt,
            min(cast(m.n_mcc as string)) as mcc_any,
            count(distinct cast(m.n_mcc as string)) as mcc_variants_cnt
        from ods_alpha.scd1_merchants m
        where m.n_cmp in ({in_list})
          and m.c_nmrc is not null
        group by m.n_cmp
    ),
    cmp_terms as (
        select distinct
            cast(m.n_cmp as string) as n_cmp_client,
            cast(t.c_nter as string) as c_nter
        from ods_alpha.scd1_merchants m
        join ods_alpha.scd1_pos_terminals t on t.c_nmrc = m.c_nmrc and t.c_nter is not null
        where m.n_cmp in ({in_list})
          and m.c_nmrc is not null
    ),
    cmp_term_agg as (
        select n_cmp_client, count(distinct c_nter) as term_cnt
        from cmp_terms
        group by n_cmp_client
    ),
    cmp_amort as (
        select
            ct.n_cmp_client,
            sum(coalesce(cast(am.amortization_monthly as double), 0.0)) as amortization
        from cmp_terms ct
        left join sandbox_ai.shestopalov_terminal_amortization_oneoff am
          on cast(am.c_nter as string) = ct.c_nter
        group by ct.n_cmp_client
    )
    select
        coalesce(r.n_cmp_client, t.n_cmp_client, a.n_cmp_client) as n_cmp_client,
        r.retl_cnt,
        r.mcc_any,
        r.mcc_variants_cnt,
        t.term_cnt,
        a.amortization
    from cmp_retl r
    full outer join cmp_term_agg t on t.n_cmp_client = r.n_cmp_client
    full outer join cmp_amort a on a.n_cmp_client = coalesce(r.n_cmp_client, t.n_cmp_client)
    """
    with imp:
        cmp_df = imp.fetch(sql_cmp)

display(cmp_df)
print('cmp_rows =', len(cmp_df))


## 4) Trx-метрики (по n_agr)


In [ ]:
n_agrs = sorted(base_df['n_agr'].dropna().unique().tolist())
if not n_agrs:
    trx_df = pd.DataFrame(columns=['n_agr'])
else:
    in_list = to_sql_in_list(n_agrs)
    sql_trx = f"""
    with trx_base as (
        select cast(t.n_trx as string) as n_trx,
               cast(t.n_amt_src as double) as n_amt_src
        from ods_alpha.scd1_trx t
        where to_date(cast(t.d_trx_orig as timestamp)) between cast('{month_start}' as date) and cast('{month_end}' as date)
          and t.c_nter is not null
          and coalesce(t.ods_deleted_flg, '0') <> '1'
    ),
    trx_acq_small as (
        select cast(ta.n_trx as string) as n_trx,
               cast(ta.n_agr as string) as n_agr,
               coalesce(cast(ta.n_amt_tax as double), 0.0) as n_amt_tax
        from ods_alpha.scd1_trx_acq ta
        where cast(ta.n_agr as string) in ({in_list})
    ),
    trx_join as (
        select ta.n_agr, ta.n_trx, tb.n_amt_src, ta.n_amt_tax
        from trx_base tb
        join trx_acq_small ta on ta.n_trx = tb.n_trx
    )
    select
        tj.n_agr,
        count(distinct tj.n_trx) as trx_cnt,
        sum(tj.n_amt_src) as trx_sum,
        sum(tj.n_amt_tax) as commission_from_ops,
        sum(coalesce(cast(i.n_amt_fee as double), 0.0)) as int_component
    from trx_join tj
    left join ods_alpha.scd1_trx_int i on cast(i.n_trx as string) = tj.n_trx
    group by tj.n_agr
    """
    with imp:
        trx_df = imp.fetch(sql_trx)

display(trx_df)
print('trx_rows =', len(trx_df))


## 5) Итоговый merge + расчетные поля


In [ ]:
attrs_df = base_df.copy()

if len(r2_df):
    attrs_df = attrs_df.merge(r2_df, on='agr_id_norm', how='left')
if len(cmp_df):
    attrs_df = attrs_df.merge(cmp_df, on='n_cmp_client', how='left')
if len(trx_df):
    attrs_df = attrs_df.merge(trx_df, on='n_agr', how='left')

for c in ['retl_cnt','term_cnt','trx_cnt','trx_sum','amortization','commission_from_ops','commission_monthly','int_component']:
    if c in attrs_df.columns:
        attrs_df[c] = pd.to_numeric(attrs_df[c], errors='coerce').fillna(0)

attrs_df['aur'] = pd.to_numeric(attrs_df.get('term_cnt', 0), errors='coerce').fillna(0) * aur_rate
attrs_df['commission_total'] = attrs_df.get('commission_from_ops', 0) + attrs_df.get('commission_monthly', 0)
attrs_df['nod_te'] = attrs_df['commission_total'] - attrs_df.get('int_component', 0)
attrs_df['fin_result_te'] = attrs_df['nod_te'] - attrs_df['aur'] - attrs_df.get('amortization', 0)
attrs_df['avg_acq_pct'] = attrs_df.apply(lambda r: (r['commission_from_ops'] / r['trx_sum'] * 100.0) if pd.notna(r['trx_sum']) and r['trx_sum'] not in (0, 0.0) else None, axis=1)
attrs_df['avg_irf_pct'] = attrs_df.apply(lambda r: (r['int_component'] / r['trx_sum'] * 100.0) if pd.notna(r['trx_sum']) and r['trx_sum'] not in (0, 0.0) else None, axis=1)

display(attrs_df)


## 6) DQ/проверка: какие поля не загрузились


In [ ]:
dq_cols = [
    'inn_norm', 'contract_number', 'agr_id_norm', 'client_name', 'ogrn',
    'filial_rf', 'vsp_name', 'vsp_code', 'tariff_name',
    'mcc_any', 'retl_cnt', 'term_cnt',
    'trx_cnt', 'trx_sum', 'commission_from_ops', 'commission_monthly', 'commission_total',
    'int_component', 'aur', 'amortization', 'nod_te', 'fin_result_te',
]

dq_rows = []
for c in dq_cols:
    if c in attrs_df.columns:
        dq_rows.append({
            'column_name': c,
            'null_rows': int(attrs_df[c].isna().sum()),
            'null_share': float(attrs_df[c].isna().mean()) if len(attrs_df) else None,
        })

dq_df = pd.DataFrame(dq_rows).sort_values(['null_share', 'null_rows'], ascending=[False, False])
display(dq_df)

display(pd.DataFrame([
    {'metric': 'rows_total', 'value': len(attrs_df)},
    {'metric': 'unique_inn_contract', 'value': attrs_df[['inn_norm', 'contract_norm']].drop_duplicates().shape[0] if len(attrs_df) else 0},
]))
